# CAPITAL ONE AIRLINE DATA CHALLENGE - PySpark Notebook

### Initialize Spark Session

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("CapitalOne_Airline_Challenge") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

print("Spark Session Created")

### 1. Load Data & Initial Quality Checks

In [0]:
display(dbutils.fs.ls("/Volumes/airline_data_challenge_pyspark_catlog/bronze/volume/"))

In [0]:
# Load datasets

# Load datasets (adjust paths)
flights = spark.read.csv("/Volumes/airline_data_challenge_pyspark_catlog/bronze/volume/Flights.csv", header=True, inferSchema=True)
tickets = spark.read.csv("/Volumes/airline_data_challenge_pyspark_catlog/bronze/volume/Tickets.csv", header=True, inferSchema=True)
airports = spark.read.csv("/Volumes/airline_data_challenge_pyspark_catlog/bronze/volume/Airport_Codes.csv", header=True, inferSchema=True)



In [0]:
# Filter to medium/large US airports

airports_us = airports.filter(
              (col("TYPE").isin(["medium_airport", "large_airport"])) &
              (col("ISO_COUNTRY") == "US")).select('IATA_CODE','TYPE').dropDuplicates()

airports_us.display()


###2. Data Cleaning & Feature Engineering

In [0]:
# Flights cleaning
flights_clean = flights.filter(col('CANCELLED') == 0 ) \
            .withColumn('AIR_TIME', expr('try_cast(AIR_TIME as double)'))\
            .withColumn('DISTANCE', expr('try_cast(DISTANCE as double)'))\
            .withColumn('OCCUPANCY_RATE', expr('try_cast(OCCUPANCY_RATE as double)'))\
            .withColumn('DEP_DELAY_CLEAN', when(col('DEP_DELAY') < 15, 0).otherwise(col('DEP_DELAY')-15))\
            .withColumn('ARR_DELAY_CLEAN', when(col('ARR_DELAY') < 15, 0).otherwise(col('ARR_DELAY')-15))\
            .withColumn('PASSENGERS', col('OCCUPANCY_RATE') * 200) #max 200 passengers

# Update rows with any nulls in key columns
flights_clean = flights_clean.fillna(0,subset=[
    'AIR_TIME', 'DISTANCE', 'OCCUPANCY_RATE', 'DEP_DELAY_CLEAN', 'ARR_DELAY_CLEAN', 'PASSENGERS', 'ORIGIN', 'DESTINATION'
])

# Create canonical route key for round trips
flights_clean = flights_clean\
    .withColumn('ROUTE_KEY', \
        when (col('ORIGIN')<col('DESTINATION'), concat(col('ORIGIN'),lit('-'), col('DESTINATION')))\
        .otherwise(concat(col('DESTINATION'),lit('-'),col('ORIGIN'))))\
    .withColumn('DIRECTION', concat(col('ORIGIN'), lit('->'), col('DESTINATION')))



# Tickets: Roundtrip only + clean fare

tickets_clean = tickets.filter(col('ROUNDTRIP') == 1)\
                .withColumn('ITIN_FARE', expr('try_cast(regexp_replace(ITIN_FARE, "[^\\d.]", "") as double)'))


print("Data cleaned. Sample route keys created.")
flights_clean.display()

###3. Aggregate at Round-Trip Route Level

In [0]:
# Aggregate Flights per route (both directions count as round-trip flights)
route_agg = flights_clean.groupBy('ROUTE_KEY')\
    .agg(
        count('*').alias("TOTAL_ROUNDTRIP_FLIGHTS"),
        sum('PASSENGERS').alias('TOTAL_PASSENGERS'),
        avg('DISTANCE').alias('AVG_DISTANCE'),
        sum('DEP_DELAY_CLEAN').alias('TOTAL_DEP_DELAY_MIN'),
        sum('ARR_DELAY_CLEAN').alias('TOTAL_ARR_DELAY_MIN') 

    )

# Join Airport types

route_agg = route_agg \
    .join(airports_us.alias("origin_airport"), 
          expr("split(ROUTE_KEY, '-')[0] = origin_airport.IATA_CODE"), "left") \
    .withColumnRenamed("TYPE", "ORIGIN_TYPE") \
    .join(airports_us.alias("dest_airport"), 
          expr("split(ROUTE_KEY, '-')[1] = dest_airport.IATA_CODE"), "left") \
    .withColumnRenamed("TYPE", "DEST_TYPE")

print("Route-level aggregation complete.")


###4. Revenue & Cost Calculations

In [0]:
# Revenue components (per the instructions)
route_agg = route_agg \
    .withColumn("AVG_TICKET_REVENUE", lit(0))\
    .withColumn("TICKET_REVENUE", col("TOTAL_PASSENGERS") * 400)\
    .withColumn("BAGGAGE_REVENUE", col("TOTAL_PASSENGERS") * 0.5 * 70) \
    .withColumn("TOTAL_REVENUE", col("TICKET_REVENUE") + col("BAGGAGE_REVENUE"))


# Costs
route_agg = route_agg \
    .withColumn("FLIGHT_MILES", col("TOTAL_ROUNDTRIP_FLIGHTS") * col("AVG_DISTANCE")) \
    .withColumn("OPERATING_COST", col("FLIGHT_MILES") * (8 + 1.18)) \
    .withColumn("AIRPORT_COST", 
                (when(col("ORIGIN_TYPE") == "large_airport", 10000).otherwise(5000) +
                 when(col("DEST_TYPE") == "large_airport", 10000).otherwise(5000)) * 
                col("TOTAL_ROUNDTRIP_FLIGHTS")) \
    .withColumn("DELAY_COST", (col("TOTAL_DEP_DELAY_MIN") + col("TOTAL_ARR_DELAY_MIN")) * 75) \
    .withColumn("TOTAL_COST", col("OPERATING_COST") + col("AIRPORT_COST") + col("DELAY_COST"))

route_agg = route_agg.withColumn("PROFIT", col("TOTAL_REVENUE") - col("TOTAL_COST"))



# route_agg.display()

###5. Top 10 Busiest & Most Profitable Routes

In [0]:
# 1. Top 10 Busiest
busiest = route_agg.orderBy(col("TOTAL_ROUNDTRIP_FLIGHTS").desc()).limit(10)
busiest.select("ROUTE_KEY", "TOTAL_ROUNDTRIP_FLIGHTS").display()

# 2. Top 10 Most Profitable
profitable = route_agg.orderBy(col("PROFIT").desc()).limit(10)
profitable.select(
    "ROUTE_KEY", "PROFIT", "TOTAL_REVENUE", "TOTAL_COST", 
    "TOTAL_ROUNDTRIP_FLIGHTS", "AVG_DISTANCE"
).display()

###6. Recommendation of 5 Routes + Breakeven

In [0]:
# Recommendation criteria (example - customize):
# High profit + high volume + reasonable delay rate
recommended = route_agg \
    .filter(col("TOTAL_ROUNDTRIP_FLIGHTS") >= 50) \
    .orderBy(col("PROFIT").desc(), col("TOTAL_ROUNDTRIP_FLIGHTS").desc()) \
    .limit(5)

recommended = recommended.withColumn(
    "BREAKEVEN_FLIGHTS", 
    ceil(lit(90_000_000) / (col("PROFIT") / col("TOTAL_ROUNDTRIP_FLIGHTS")))
)

recommended.select(
    "ROUTE_KEY", "PROFIT", "TOTAL_ROUNDTRIP_FLIGHTS", 
    "BREAKEVEN_FLIGHTS", "AVG_DISTANCE"
).display()